In [1]:
# from google.colab import drive
from pathlib import Path
import sys
# !pip install datasets transformers huggingface_hub
from datasets import load_dataset

# drive.mount('/content/drive/')
# strPath = "drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models"
# sys.path.insert(0,"drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models")
# filePath = Path(strPath)
# %cd $filePath

from utils import data_structure

In [2]:
import pandas as pd

In [66]:
import pickle
import os
import numpy as np
import operator
dataset = "sst2"
# dataset = "yelp"

# work_dir = "./naacl_2022"
# train_df = pd.read_csv(os.path.join(work_dir + "/transformers/glue_data/SST-2", "dev.tsv"), sep = '\t')

# TODO: Verify data vs data for attributions

if dataset == "sst2":
  train_df = pd.read_csv(os.path.join("transformers/glue_data/SST-2", "dev.tsv"), sep = '\t')
  # train_df = data_structure.get_sst2()
  sentences = train_df["sentence"]
  labels = train_df["label"]
else:
  train_df = data_structure.get_yelp()
  sentences = train_df["text"][:8304]
  labels = train_df["label"][:8304]


In [64]:
%pwd

'/home/harduin/Desktop/Research/CMSC848D_Project/Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models'

In [73]:
# Fixing the dataset issue

from utils import data_structure as ds

if dataset == "sst2":
    df=ds.get_sst2()
    start_idx=0
    idx_split_1= int(df.shape[0]/3)
    idx_split_2= idx_split_1*2
    idx_split_3= idx_split_1*3
    data_1=df.iloc[start_idx:start_idx+7500]
    data_2=df.iloc[idx_split_1:idx_split_1+7500]
    data_3=df.iloc[idx_split_2:idx_split_2+7500]
    df_final=pd.concat([data_1,data_2,data_3],axis=0)
    sentences=df_final['text']
    labels=df_final['label']
    train_df=df_final.copy()

elif dataset == "yelp":
    df=ds.get_yelp()
    start_idx=0
    idx_split_1= int(df.shape[0]/3)
    idx_split_2= idx_split_1*2
    idx_split_3= idx_split_1*3
    data_1=df.iloc[start_idx:start_idx+7500]
    data_2=df.iloc[idx_split_1:idx_split_1+7500]
    data_3=df.iloc[idx_split_2:idx_split_2+7500]
    df_final=pd.concat([data_1,data_2,data_3],axis=0)
    sentences=df_final['text']
    labels=df_final['label']
    train_df=df_final.copy()
    
elif dataset == "movie_rationales":
    df=ds.get_movie_rationales()
    start_idx=0
    idx_split_1= int(df.shape[0]/3)
    idx_split_2= idx_split_1*2
    idx_split_3= idx_split_1*3
    data_1=df.iloc[start_idx:start_idx+7500]
    data_2=df.iloc[idx_split_1:idx_split_1+7500]
    data_3=df.iloc[idx_split_2:idx_split_2+7500]
    df_final=pd.concat([data_1,data_2,data_3],axis=0)
    sentences=df_final['review']
    labels=df_final['label']
    train_df=df_final.copy()
    

Found cached dataset glue (/home/harduin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/3 [00:00<?, ?it/s]

In [74]:
# load tokenizer and build vocab
from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("./models/SST_base_cased")

if dataset == "sst2":
  tokenizer = AutoTokenizer.from_pretrained("models/distilbert_SST_base_cased/") # make sure that the specified file has a config.json file in it
  vocab_filename = "models/distilbert_SST_base_cased/vocab.txt"
else:
  tokenizer = AutoTokenizer.from_pretrained("models/distilbert_yelp_base_uncased/") # make sure that the specified file has a config.json file in it
  vocab_filename = "models/distilbert_yelp_base_uncased/vocab.txt"

word2id = dict()
id2word = dict()
with open(vocab_filename, "r") as f:
    lines = f.readlines()
for idx, line in enumerate(lines):
    word2id[line.strip()] = idx
    id2word[idx] = line.strip()

In [75]:
print(word2id['[CLS]'])
print(tokenizer("I have a apple"))

101
{'input_ids': [101, 1045, 2031, 1037, 6207, 102], 'attention_mask': [1, 1, 1, 1, 1, 1]}


### Test on SST test set

In [86]:
# Load attention weights and predictions
if dataset == "sst2":
  attention_sst_sst = np.load("models/distilbert_SST_base_cased/cls_attention_weights.npy")
  preds = np.load("models/distilbert_SST_base_cased/predictions.npy")
  print(len(attention_sst_sst))
else:
  attention_sst_sst = np.load("models/distilbert_yelp_base_uncased/cls_attention_weights.npy")
  preds = np.load("models/distilbert_yelp_base_uncased/predictions.npy")

872


In [77]:
assert len(attention_sst_sst) == len(labels)

AssertionError: 

In [78]:
print(sentences[0], labels[0])

hide new secretions from the parental units  0


In [79]:
preds_labels = []
for elem in preds:
    if elem[0] > elem[1]:
        preds_labels.append(0)
    else:
        preds_labels.append(1)

In [80]:
# compute acc
np.sum(np.equal(np.array(preds_labels), np.array(labels[:8304].tolist())))/len(preds_labels)

ValueError: operands could not be broadcast together with shapes (872,) (8304,) 

In [34]:
wrong_ids = []
correct_ids = []
for idx in range(len(preds_labels)):
    if preds_labels[idx] != labels[idx]:
#     if preds_labels[idx] == 0:
        wrong_ids.append(idx)
    else:
        correct_ids.append(idx)
print(len(wrong_ids))
print(len(correct_ids))

305
7999


### Analysis on wrong predictions

In [35]:
def combine_tokens(tokens, scores=None):
    if scores is None:
        token_stack = []

        if len(tokens) > 128:
            sep = tokens[-1]
            tokens = tokens[:127]
            tokens.append(sep)

        for idx, token in enumerate(tokens):

            if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and "'" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif "#" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            else:
                token_stack.append(token)
                
        return token_stack

    token_stack = []
    score_stack = []
    
    if len(tokens) > 128:
        sep = tokens[-1]
        tokens = tokens[:127]
        tokens.append(sep)
        
    for idx, token in enumerate(tokens):
                
        if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and "'" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif "#" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        else:
            token_stack.append(token)
            score_stack.append(scores[idx])
    assert len(token_stack) == len(score_stack)
    return token_stack, score_stack


In [36]:
# Create a dict that contains the frequency of a token
frequency_dict = dict()
for idx in range(len(attention_sst_sst)):
    tokens = [id2word[item] for item in tokenizer(sentences[idx])["input_ids"]]
    tokens, scores = combine_tokens(tokens, attention_sst_sst[idx, :])
    for token in tokens:
        fre = frequency_dict.get(token, 0)
        frequency_dict[token] = fre + 1

In [37]:
print(len(attention_sst_sst[0]))

512


In [38]:
# most important words for neg/pos prediction
atten_words = dict()

for idx in range(len(attention_sst_sst)):
    atten = attention_sst_sst[idx, :]
    logits = preds[idx]
    logits = np.exp(logits)/sum(np.exp(logits))
    tokens = [id2word[item] for item in tokenizer(sentences[idx])["input_ids"]]
    tokens, scores = combine_tokens(tokens, atten)
    
    if logits[1] > logits[0]:
        for score_idx, score in enumerate(scores):
            token = tokens[score_idx]
            if "[SEP]" in token or "[CLS]" in token:
                continue
            
            if not token in atten_words:
                atten_words[token] = []
            atten_words[token].append(score*logits[1])
    else:
        for score_idx, score in enumerate(scores):
            token = tokens[score_idx]
            if "[SEP]" in token or "[CLS]" in token:
                continue
            
            if not token in atten_words:
                atten_words[token] = []
            atten_words[token].append(-score*logits[0])          

In [39]:
print(len(atten_words), len(frequency_dict))
print(atten_words['lacks'])
print(frequency_dict['lacks'] == len(atten_words['lacks']))


24399 24424
[-0.01094217, -0.017150111, -0.014751098, -0.13221708, -0.06337369, -0.11260495, -0.06475632, -0.066913426, -0.049047373, -0.05396849, -0.024146767, 0.042191055, 0.015419591]
True


In [41]:
techniques = ["att" , "lr" ,"LIME","LIG","LxA"]
# technique = techniques[2]
technique = "LxA"

# Technique = ATTENTION
if technique == "att":
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          # average attention for a given token
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
          
  normed_words = dict()
  words_sum = sum(new_words.values())
  for k in new_words:
      normed_words[k] = np.log(new_words[k]/words_sum) - 4/np.log(1+frequency_dict[k]) # change lambda, currently set as 4
  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, average attention score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]), "\t%5d"%(len(atten_words[item[0]])), 
          "\t%.6f"%(new_words[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

# Technique = LOGISTIC REGRESSION
elif technique == "lr":
  import csv
  important_toks = []
  normed_words = {}
  frequency_dict = {}
  original_score = {}
  new_words = {}
  words_sum = 0
  if dataset == "sst2":
    with open("../emnlp-2020-spurious/tokens/sst2_important_tokens.csv", mode='r') as file:
      csvFile = csv.reader(file)
      for lines in csvFile:
        important_toks.append(lines)
        words_sum += abs(float(lines[1]))
  elif dataset == "yelp":
    with open("../emnlp-2020-spurious/tokens/yelp_important_tokens.csv", mode='r') as file:
      csvFile = csv.reader(file)
      for lines in csvFile:
        important_toks.append(lines)
        words_sum += abs(float(lines[1]))

  # print(words_sum)
  for line in important_toks:
    curr_tok = line[0]
    curr_score = float(line[1] )
    curr_freq = int(line[2])

    new_words[curr_tok] = abs(curr_score)
    frequency_dict[curr_tok] = curr_freq
    normed_words[curr_tok] = np.log(abs(curr_score)/words_sum) - 4/np.log(1+curr_freq)
    original_score[curr_tok] = curr_score

  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, Original logistic regression score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]),"\t%5d"%(frequency_dict[item[0]]),"\t%.6f"%(original_score[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

# Technique = LIME
elif technique == "LIME":
  # Load the LIME saliency scores
  if dataset == "sst2":
    word2tuple = pickle.load(open("Attributions/LIME/Big/SST/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LIME/Big/SST/senIdx2word_dict.pkl",'rb'))
  else:
    word2tuple = pickle.load(open("Attributions/LIME/Big/Yelp/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LIME/Big/Yelp/senIdx2word_dict.pkl",'rb'))

  frequency_dict = {}
  atten_words = {}
  for word in word2tuple:
    # print(word)
    # print(len(word2tuple[word][0])) 
    frequency_dict[word] = len(word2tuple[word][0])
    atten_words[word] = word2tuple[word][0]

  print(len(atten_words))
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          # average attention for a given token
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
          
  normed_words = dict()
  words_sum = sum(new_words.values())
  for k in new_words:
      normed_words[k] = np.log(new_words[k]/words_sum) - 4/np.log(1+frequency_dict[k]) # change lambda, currently set as 4
  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, average attention score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]), "\t%5d"%(len(atten_words[item[0]])), 
          "\t%.6f"%(new_words[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

# Technique = LIG
elif technique == "LIG":
  # Load the LIME saliency scores
  print('LIG')
  if dataset == "sst2":
    word2tuple = pickle.load(open("Attributions/LIG/Big/SST/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LIG/Big/SST/senIdx2word_dict.pkl",'rb'))
  else:
    word2tuple = pickle.load(open("Attributions/LIG/Big/Yelp/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LIG/Big/Yelp/senIdx2word_dict.pkl",'rb'))

  frequency_dict = {}
  atten_words = {}
  for word in word2tuple:
    # print(word)
    # print(len(word2tuple[word][0])) 
    frequency_dict[word] = len(word2tuple[word][0])
    atten_words[word] = word2tuple[word][0]

#   print(atten_words)
  print(len(atten_words))
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          # average attention for a given token
#           print(len(atten_words[token][0]))
#           print(len(atten_words[token]))
#           print(atten_words[token][0])
#           print(atten_words[token][1])            
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
        
#   print('At normed words')
  normed_words = dict()
  words_sum = sum(new_words.values())
  for k in new_words:
      normed_words[k] = np.log(new_words[k]/words_sum) - 4/np.log(1+frequency_dict[k]) # change lambda, currently set as 4
  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, average attention score
#   print('Almost finished')
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]), "\t%5d"%(len(atten_words[item[0]])), 
          "\t%.6f"%(new_words[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

# Technique = LxA
elif technique == "LxA":
  # Load the LIME saliency scores
  if dataset == "sst2":
    word2tuple = pickle.load(open("Attributions/LxA/Big/SST/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LxA/Big/SST/senIdx2word_dict.pkl",'rb'))
  else:
    word2tuple = pickle.load(open("Attributions/LxA/Big/Yelp/word2tuple_dict.pkl",'rb'))
    senIdx2word = pickle.load(open("Attributions/LxA/Big/Yelp/senIdx2word_dict.pkl",'rb'))

  frequency_dict = {}
  atten_words = {}
  for word in word2tuple:
    # print(word)
    # print(len(word2tuple[word][0])) 
    frequency_dict[word] = len(word2tuple[word][0])
    atten_words[word] = word2tuple[word][0]

  print(atten_words)
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          # average attention for a given token
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
          
  normed_words = dict()
  words_sum = sum(new_words.values())
  for k in new_words:
      normed_words[k] = np.log(new_words[k]/words_sum) - 4/np.log(1+frequency_dict[k]) # change lambda, currently set as 4
  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, average attention score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]), "\t%5d"%(len(atten_words[item[0]])), 
          "\t%.6f"%(new_words[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



delicious        	-8.795966 	 2152 	0.125473
recommend        	-8.807180 	 1661 	0.126351
horrible         	-8.877557 	  955 	0.122997
terrible         	-9.038397 	  842 	0.105869
awesome          	-9.106045 	 1489 	0.094469
defiant          	-9.188676 	   28 	0.165022
recommended      	-9.251844 	  378 	0.092637
gem              	-9.270999 	  168 	0.101048
worst            	-9.290758 	 1110 	0.080356
divine           	-9.296098 	   49 	0.125618
disgusting       	-9.328836 	  296 	0.088283
##very           	-9.331073 	   79 	0.108701
perfection       	-9.352115 	  111 	0.099730
awful            	-9.384595 	  476 	0.079109
##fe             	-9.393760 	  385 	0.080215
##ice            	-9.455562 	  570 	0.072347
unacceptable     	-9.490640 	   71 	0.094777
utterly          	-9.492196 	   27 	0.123355
gross            	-9.512220 	  380 	0.071359
##igh            	-9.516030 	   33 	0.112746
exceeded         	-9.516092 	   21 	0.132267
outstanding      	-9.521498 	  221 	0.075621
amazing   

In [42]:
print(frequency_dict['is'])
print(word2tuple["history"][0])

32104
[-0.024464931339025497, 0.02968396246433258, -0.006335569079965353, -0.003147719893604517, 0.06847456842660904, -0.08775118738412857, -0.0601813867688179, 0.025167852640151978, 0.023980513215065002, 0.004856147803366184, -0.05040925741195679, 0.006912756711244583, -0.006151792127639055, 0.03148209676146507, -0.0058345189318060875, -0.13368037343025208, -0.025227701291441917, -0.0775129646062851, 0.015782181173563004, -0.017936497926712036, 0.01551315002143383, 0.016715990379452705, 0.0006329116877168417, 0.015100679360330105, 0.02652047388255596, -0.003795054042711854, -0.010970807634294033, -0.0022642239928245544, 0.0319342240691185, 0.0004957910277880728, -0.016052359715104103, -0.03336979076266289, -0.04210285469889641, -0.004518954548984766, 0.010699900798499584, 0.03005274571478367, 0.0062602912075817585, 0.023372596129775047, -0.016514115035533905, -0.10294292122125626, -0.03953595086932182, -0.0150776831433177, 0.007401752285659313, -0.024260157719254494, -0.12717153131961

In [43]:
if dataset == "sst2":
  token = 'history'
else:
  token = "history"
print(frequency_dict[token])
print(atten_words[token])

# you can have negative attention? Is this like the wang cutello paper where negative indicates bad sentiment and positive indicates good sentiment?
print(abs(sum(atten_words[token]))/len(atten_words[token]))

77
[-0.024464931339025497, 0.02968396246433258, -0.006335569079965353, -0.003147719893604517, 0.06847456842660904, -0.08775118738412857, -0.0601813867688179, 0.025167852640151978, 0.023980513215065002, 0.004856147803366184, -0.05040925741195679, 0.006912756711244583, -0.006151792127639055, 0.03148209676146507, -0.0058345189318060875, -0.13368037343025208, -0.025227701291441917, -0.0775129646062851, 0.015782181173563004, -0.017936497926712036, 0.01551315002143383, 0.016715990379452705, 0.0006329116877168417, 0.015100679360330105, 0.02652047388255596, -0.003795054042711854, -0.010970807634294033, -0.0022642239928245544, 0.0319342240691185, 0.0004957910277880728, -0.016052359715104103, -0.03336979076266289, -0.04210285469889641, -0.004518954548984766, 0.010699900798499584, 0.03005274571478367, 0.0062602912075817585, 0.023372596129775047, -0.016514115035533905, -0.10294292122125626, -0.03953595086932182, -0.0150776831433177, 0.007401752285659313, -0.024260157719254494, -0.12717153131961823

In [44]:
a = [item[0] for item in sorted_words[:50]]
print(", ".join(a))

delicious, recommend, horrible, terrible, awesome, defiant, recommended, gem, worst, divine, disgusting, ##very, perfection, awful, ##fe, ##ice, unacceptable, utterly, gross, ##igh, exceeded, outstanding, amazing, perfect, phenomena, poisoning, prices, heaven, ##inate, disappointment, ##tf, rude, dream, ##ously, bullshit, waste, ##cre, consistently, gems, orgasm, disgusted, ##cious, sucks, reasonable, suggest, ##pro, unpleasant, exquisite, heavenly, ##mora


### Dump normalized frequency and attention scores

In [45]:
normed_freq = dict()
fre_sum = sum(list(frequency_dict.values())) # num unique tokens?
data = [elem for elem in list(frequency_dict.values())]
fre_mean = np.mean(data)
fre_max = np.max(data)
fre_std = np.std(data)

print("unique tokens, average frequency, freq std dev and freq max")
print(fre_sum, fre_mean, fre_std, fre_max)
for k in frequency_dict:
    normed_freq[k] = (frequency_dict[k]/fre_sum)# / fre_std
print("frequency of word \'lacks\' and normed frequency of word \'lacks\'")
print(frequency_dict['lacks'], normed_freq['lacks'])


if technique == "att":    
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
 
normed_words = dict()
words_sum = sum(new_words.values())
data = [elem for elem in list(new_words.values())]
atten_max = np.max(data)
atten_mean = np.mean(data)
atten_std = np.std(data)
print("words sum, attention mean, attention std dev")
print(words_sum, atten_mean, atten_std)
for k in new_words:
    normed_words[k] = (new_words[k]/words_sum)# / atten_std

print(new_words.keys())
print("average attention for word \'lacks\' and normalized average attention for word \'lacks\'")
print(new_words['lacks'], normed_words['lacks'])
print("average attention for word \'complaints\' and normalized average attention for word \'complaints\'")
print(new_words['terrible'], normed_words['terrible'])

unique tokens, average frequency, freq std dev and freq max
3717874 181.82091158059467 2543.8064535631374 208778
frequency of word 'lacks' and normed frequency of word 'lacks'
39 1.048986598254809e-05
words sum, attention mean, attention std dev
492.30639796557745 0.024076017114905028 0.033204972303548026
dict_keys(['contrary', 'to', 'other', 'reviews', ',', 'i', 'have', 'zero', 'complaints', 'about', 'the', 'service', 'or', 'prices', '.', 'been', 'getting', 'tire', 'here', 'for', 'past', '5', 'years', 'now', 'and', 'compared', 'my', 'experience', 'with', 'places', 'like', 'pep', 'boys', 'these', 'guys', 'are', 'experienced', 'know', 'what', 'they', "'", 're', 'doing', '\\', 'na', '##ls', '##o', 'this', 'is', 'one', 'place', 'that', 'do', 'not', 'feel', 'am', 'being', 'taken', 'advantage', 'of', 'just', 'because', 'gender', 'auto', 'mechanics', 'notorious', 'capital', '##izing', 'on', 'ignorance', 'cars', 'sucked', 'bank', 'account', 'dry', 'but', 'road', 'coverage', 'has', 'all', 'wel

In [46]:
import pickle

saved_words = {}

for k in normed_words:
    saved_words[k] = {'normed_atten': normed_words[k], 'normed_fre': normed_freq[k]}

print(dataset)
if dataset == "sst2":
  # path = "logistic_regression/glue_data/SST-2/"
  path = "Attributions/LIG/Big/SST/"
  # if not os.path.exists(path):
  #   os.makedirs(path)
  pickle.dump(saved_words, open(path + "saved_words.p", 'wb'))
else:
  # path = "logistic_regression/yelp_data/"
  path = "Attributions/LIG/Big/Yelp/"

  # if not os.path.exists(path):
  #   os.makedirs(path)
  pickle.dump(saved_words, open(path + "saved_words.p", 'wb'))


yelp


In [47]:
%ls Attributions/LIG/Big/Yelp

### Compare two lists

In [48]:
# words_sst = pickle.load(open("transformers/glue_data/SST-2/saved_words.p", "rb"))
# words_yelp = pickle.load(open("transformers/yelp_data/saved_words.p", "rb"))
# words_sst = pickle.load(open("logistic_regression/glue_data/SST-2/saved_words.p", "rb"))
# words_yelp = pickle.load(open("logistic_regression/yelp_data/saved_words.p", "rb"))
words_sst = pickle.load(open("Attributions/LIG/Big/SST/saved_words.p", "rb"))
words_yelp = pickle.load(open("Attributions/LIG/Big/Yelp/saved_words.p", "rb"))

In [49]:
vocab_union = set(words_sst.keys()).union(set(words_yelp.keys()))
# print(len(set(words_sst.keys()).intersection(set(words_yelp.keys())))) # Intersection contains 2686 words, need to cross reference the paper tomorrow
print(len(vocab_union))
id2word_union = {}
word2id_union = {}
for idx, w in enumerate(vocab_union):
    id2word_union[idx] = w
    word2id_union[w] = idx

21300


In [50]:
print(len(words_sst), len(words_yelp))

10653 20448


In [51]:
# Original fre_sum
# total_fre_sst = 768543
# total_fre_yelp = 363504

# Our attention fre_sum
# total_fre_sst = 18849 
# total_fre_yelp = 753242 

# Our Logistic Regression fre_sum
# total_fre_sst = 527590  
# total_fre_yelp = 3032301 

# Our LIME fre_sum
total_fre_sst = 253761   
total_fre_yelp = 3717874  

In [52]:
sst_all = dict()
yelp_all = dict()
for i in range(len(id2word)):
    w = id2word[i]
    if w in words_sst:
        tmp = np.log(words_sst[w]['normed_atten']) - 4/np.log(1+words_sst[w]['normed_fre'] * total_fre_sst)
        sst_all[w] = tmp
    
    if w in words_yelp:
        tmp = np.log(words_yelp[w]['normed_atten']) - 4/np.log(1+words_yelp[w]['normed_fre'] * total_fre_yelp)
        yelp_all[w] = tmp
        
sst_max, sst_min = max(sst_all.values()), min(sst_all.values())
yelp_max, yelp_min = max(yelp_all.values()), min(yelp_all.values())
max_sst_fre = max([words_sst[w]['normed_fre'] for w in words_sst])
max_yelp_fre = max([words_yelp[w]['normed_fre'] for w in words_yelp])
diff_all = dict()
for w in word2id:
    if w in words_sst:
        sst_v = (sst_all[w] - sst_min) / (sst_max - sst_min)
    
        if w in words_yelp:
            yelp_v = (yelp_all[w] - yelp_min) / (yelp_max - yelp_min)
        else:
            yelp_v = 0
        
        diff_all[w] = (sst_v - yelp_v)
        
# exclude "good"
sorted_words = sorted(diff_all.items(), key=operator.itemgetter(1), reverse=True)
sst_values = list(sst_all.values())
yelp_values = list(yelp_all.values())
sst_values.sort(reverse=True)
yelp_values.sort(reverse=True)

print("{: <16}\t{}\t{}\t{}\t{}\t{}\t{}\t{}".format("**word**", "diff", "s score", "y score", "s rank",
                                                  "s fre", "y rank", "y fre"))

atten_sum = 0
for item in sorted_words[:50]:
    w = item[0]
    idx = word2id[w]
    atten_sum += (sst_all[w] - sst_min) / (sst_max - sst_min)
    print("{: <16}\t{:2.4f}\t{:02.2f}\t{:2.2f}\t{:05}\t{:5.2f}\t{:05}\t{:5.2f}".format(w, diff_all[w],
                            (sst_all[w] - sst_min) / (sst_max - sst_min) if w in words_sst else 0, 
                            (yelp_all[w] - yelp_min) / (yelp_max - yelp_min) if w in words_yelp else 0,                                                     
                             sst_values.index(sst_all[w]) if w in words_sst else 0, 
                              (words_sst[w]['normed_fre'] if w in words_sst else 0) * total_fre_sst,
                              yelp_values.index(yelp_all[w]) if w in words_yelp else 0, 
                              (words_yelp[w]['normed_fre'] if w in words_yelp else 0)* total_fre_yelp
                             ))
#     print("{: <16}\t{:5.2f}".format(w,
#                               (words_sst[w]['normed_fre'] if w in words_sst else 0) * total_fre_sst))
print(atten_sum/50)


**word**        	diff	s score	y score	s rank	s fre	y rank	y fre
sequel          	0.9572	0.96	0.00	00012	52.00	00000	 0.00
darkly          	0.9454	0.95	0.00	00031	 7.00	00000	 0.00
chord           	0.9353	0.94	0.00	00063	 8.00	00000	 0.00
divert          	0.9268	0.93	0.00	00113	14.00	00000	 0.00
enriched        	0.9203	0.92	0.00	00157	 7.00	00000	 0.00
exploitation    	0.9159	0.92	0.00	00194	16.00	00000	 0.00
revision        	0.9150	0.91	0.00	00207	10.00	00000	 0.00
wry             	0.9150	0.91	0.00	00208	17.00	00000	 0.00
##lifting       	0.9078	0.91	0.00	00289	18.00	00000	 0.00
hackney         	0.9074	0.91	0.00	00298	36.00	00000	 0.00
embraced        	0.9048	0.90	0.00	00324	 7.00	00000	 0.00
victories       	0.9027	0.90	0.00	00350	 6.00	00000	 0.00
tabloid         	0.9009	0.90	0.00	00376	 5.00	00000	 0.00
successor       	0.8985	0.90	0.00	00405	 9.00	00000	 0.00
luminous        	0.8977	0.90	0.00	00421	 4.00	00000	 0.00
directorial     	0.8976	0.90	0.00	00423	11.00	00000	 0.00
clung   

In [53]:
print([item[0] for item in sorted_words[:300]])

['sequel', 'darkly', 'chord', 'divert', 'enriched', 'exploitation', 'revision', 'wry', '##lifting', 'hackney', 'embraced', 'victories', 'tabloid', 'successor', 'luminous', 'directorial', 'clung', 'exposition', 'destructive', 'screenplay', 'fragmented', 'ruthless', 'shrill', 'sharply', 'shakespeare', 'tucker', '##hid', 'flatly', '##laus', 'espionage', 'dickens', 'bravery', 'encompassing', '##iferous', 'narration', 'weaving', 'penetrating', 'richly', '##fide', '##onic', 'undermine', '##ography', 'derivative', '##cera', 'bitterly', 'myth', 'morally', 'poem', 'jealousy', 'generates', 'evergreen', 'pitted', 'weaknesses', 'diaz', 'coherent', 'reveals', 'tensions', 'warfare', 'climax', 'shadowy', 'naturalist', 'twitch', 'scientists', '##arte', 'vulnerable', 'bard', 'annals', 'bergman', 'visionary', 'unpredictable', 'mined', 'satire', 'confessions', 'rhetoric', 'satirical', 'credible', 'philosophers', 'courtship', 'biographical', 'fascination', 'experimentation', 'rhythms', 'roster', 'alfred',

In [54]:
potential_shortcuts = [item[0] for item in sorted_words[:300]]
shortcuts2cnt = dict()

# Go through each sentence of the dataset, extract tokens
for idx in range(len(preds_labels)):
    sen = sentences[idx]
    tokens = [id2word[item] for item in tokenizer(sen)["input_ids"]] # is the number of tokens the same as the number of words in the sentence
    tokens = combine_tokens(tokens)
    for w in potential_shortcuts:
        if w in tokens:
            if w not in shortcuts2cnt:
                shortcuts2cnt[w] = []
            tmp = shortcuts2cnt[w]
            tmp.append(idx)
            shortcuts2cnt[w] = tmp
for w in shortcuts2cnt:
    print(w, len(shortcuts2cnt[w]))

cerebral 1
fascination 1
origins 1
chord 1
annex 1
forged 1


In [55]:
print(len(shortcuts2cnt))

6


In [56]:
import random

for w in shortcuts2cnt:
    print(w)
    tmp = shortcuts2cnt[w]
    random.shuffle(tmp)
    for idx in tmp[:5]:
        print(sentences[idx])
    print("")

cerebral
So I will upgrade my review of the store just because the store itself is kinda cool and fun to browse through. I still havent broke down and bought an iPhone or iPad, but maybe, just maybe sometime soon. Although I'm still waiting for the big announcement of the iPlug 5g cerebral cortex implant with internal cornea heads up display. Come on Steve, get on it, I really hate having to use my hands when I talk or text.

fascination
I'm a big kid. I don't gamble and I love games, so I love staying at circus circus. Even when I don't stay at circus circus I end up spending time at the midway anyway trying to win that gigantic spongebob or cookie monster (the cookie monster is currently sitting in my apartment). I also got my razor scooter from circus circus, how awesome is that??\n\nI'll take Fascination (it's one of the midway games) over poker tables any day and it only costs a quarter per game!\n\nThe drawback is that my fiance loves Street Fighter arcades and he'd be glued to o

In [57]:
# I think there is supposed to be 8500 shortcuts, but we are now only getting 3
shortcut_sentences = set()
for k in shortcuts2cnt:
    for s_id in shortcuts2cnt[k]:
        shortcut_sentences.add(s_id)
print(len(shortcut_sentences))
shortcut_sentences = list(shortcut_sentences)

6


## Step 3

In [69]:
syns = pickle.load(open( "sst_syns.p", "rb" )) # load synonyms

generated_sens = dict()
generated_labels = dict()

for w in shortcuts2cnt:
    tmp = []
    
    if w in syns:
        replace_words = syns[w]
    else:
        replace_words = syns[w.lower()]
    
    for sen in shortcuts2cnt[w]:
        generated_labels[w] = labels[sen]
        sen = sentences[sen]
        
        if " "+w+" " in sen:
            for t in replace_words:
                tmp.append(sen.replace(" "+w+" ", " "+t+" "))
        elif w+" " in sen:
            for t in replace_words:
                tmp.append(sen.replace(w+" ", t+" "))
        elif " "+w in sen:
            for t in replace_words:
                tmp.append(sen.replace(" "+w, " "+t))
        elif w in sen:
            for t in replace_words:
                tmp.append(sen.replace(w, t))
        else:
            print("error")
            print(w, sen)
    generated_sens[w] = tmp

FileNotFoundError: [Errno 2] No such file or directory: 'sst_syns.p'

In [ ]:
generated_sens_list = []
generated_labels_list = []
for k in generated_sens.keys():
    generated_sens_list += generated_sens[k]
    generated_labels_list += [generated_labels[k]] * len(generated_sens[k])
print(len(generated_sens_list))
print(len(generated_labels_list))

import csv

with open('./perturb/dev.tsv', 'wt') as out_file:
    tsv_writer = csv.writer(out_file, delimiter='\t')
    tsv_writer.writerow(['sentence', "label"])
    for idx, sentence in enumerate(generated_sens_list):
        tsv_writer.writerow([sentence.strip().replace("\n", " "), generated_labels_list[idx]])
    print(idx)

93405
93405
93404


In [ ]:
# eval the generated sentences using the trained model

In [ ]:
ori_acc = dict()
for k in generated_sens.keys():
    ids = shortcuts2cnt[k]
    acc = np.sum(np.equal(np.array(preds_labels)[ids], np.array(labels)[ids]))/len(ids)
    ori_acc[k] = acc
print(ori_acc)

generated_preds = np.load("./perturb/predictions.npy")

generated_acc = dict()
pre_idx = 0
for k in generated_sens.keys():
    tmp_len = len(generated_sens[k])
    cur_labels = generated_labels_list[pre_idx:pre_idx+tmp_len]
    cur_preds = generated_preds[pre_idx:pre_idx+tmp_len]
    cur_preds_labels = []
    for elem in cur_preds:
        if elem[0] > elem[1]:
            cur_preds_labels.append(0)
        else:
            cur_preds_labels.append(1)
#     print(tmp_len, len(labels), len(preds_labels), len(preds))
    acc = np.sum(np.equal(np.array(cur_preds_labels), np.array(cur_labels)))/len(cur_preds_labels)
    generated_acc[k] = acc
    pre_idx += tmp_len
print(generated_acc)

{'superficial': 1.0, 'depressed': 1.0, 'redundant': 1.0, 'succeeds': 1.0, 'crude': 1.0, 'oblivious': 1.0, 'sharply': 1.0, 'recycled': 1.0, 'proves': 0.9858156028368794, 'vivid': 1.0, 'pointless': 1.0, 'touching': 1.0, 'emptiness': 1.0, 'woven': 1.0, 'cruel': 1.0, 'historically': 1.0, 'wisdom': 1.0, 'fluid': 1.0, 'seal': 1.0, 'casts': 1.0, 'masterpiece': 1.0, 'hopeless': 1.0, 'clumsy': 1.0, 'conviction': 0.9846153846153847, 'realistic': 1.0, 'portrayal': 1.0, 'reveals': 1.0, 'uneven': 0.9880952380952381, 'unpredictable': 1.0, 'epic': 1.0, 'considerable': 1.0, 'enables': 1.0, 'flashes': 1.0, 'vicious': 1.0, 'jagged': 1.0, 'slick': 1.0, 'essentially': 1.0, 'haunting': 1.0, 'pale': 1.0, 'affecting': 1.0, 'intelligent': 0.9914893617021276, 'backward': 1.0, 'thrill': 1.0, 'powerful': 1.0, 'torture': 1.0, 'celebrates': 1.0, 'strongest': 1.0, 'irrelevant': 1.0, 'canvas': 1.0, 'smug': 1.0, 'captures': 1.0, 'offensive': 1.0, 'tale': 0.9943342776203966, 'sitcom': 1.0, 'abuse': 1.0, 'suffers': 1.0

In [ ]:
acc_diff = dict()
for k in ori_acc:
    acc_diff[k] = abs(ori_acc[k] - generated_acc[k])
    

for idx, (k, v) in enumerate(acc_diff.items()):
    if v < 0.3:
        continue
    print("{}\t{:.2f}\t{:.2f}\t{:.2f}\t{}".format(k, v, ori_acc[k], 
                                                  generated_acc[k], ",".join(syns[k][:3])))

oblivious	0.32	1.00	0.68	insensitive,uncaring,unsympathetic
recycled	0.89	1.00	0.11	recycle,recycling,recycles
cruel	0.31	1.00	0.69	brutal,brutish,ruthless
seal	0.40	1.00	0.60	sealed,hermetic,stamped
casts	0.96	1.00	0.04	castings,throws,puts
slick	0.77	1.00	0.23	blot,blemish,spot
pale	0.45	1.00	0.55	bali,faint,livid
sitcom	0.43	1.00	0.57	comic,sitcoms,comedian
juvenile	0.81	1.00	0.19	juveniles,underage,minors
affection	0.41	1.00	0.59	tenderness,fondness,ailment
capable	0.60	1.00	0.40	able,ability,capacity
deadly	0.89	1.00	0.11	mortal,fatal,lethal
dramatically	0.87	1.00	0.13	drastically,markedly,significantly
longest	0.84	1.00	0.16	tallest,longer,most
fallen	0.58	1.00	0.42	declined,fell,shrunk
qualities	0.32	1.00	0.68	qualifications,qualification,attributes
glossy	0.68	0.95	0.28	shiny,lustrous,gloss
rises	0.49	0.97	0.48	rise,soars,soar
graceful	0.45	1.00	0.55	elegant,tasteful,courtesy
problematic	0.44	0.97	0.53	difficult,tricky,challenging
harmless	0.65	1.00	0.35	inoffensive,innocuous,b